# Ragas with Azure AI Search + Azure OpenAI

This notebook evaluates a RAG flow that retrieves contexts from an Azure AI Search index and uses Azure OpenAI for answering plus Ragas judging/embeddings.

Fill the environment variables in the first code cell before running all cells.

In [ ]:
import os
import requests
import nest_asyncio

nest_asyncio.apply()

# Optional: uncomment and edit these, or set them in your .env / system environment.
# os.environ["AZURE_SEARCH_ENDPOINT"] = "https://proj-default-search1.search.windows.net"
# os.environ["AZURE_SEARCH_INDEX_NAME"] = "<your-search-index-name>"
# os.environ["AZURE_SEARCH_API_KEY"] = "<your-search-admin-or-query-key>"
# os.environ["AZURE_OPENAI_ENDPOINT"] = "https://<your-azure-openai-resource>.openai.azure.com/"
# os.environ["AZURE_OPENAI_API_KEY"] = "<your-azure-openai-key>"
# os.environ["OPENAI_API_VERSION"] = "2024-02-15-preview"
# os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"] = "<your-chat-deployment-name>"
# os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"] = "<your-embedding-deployment-name>"

AZURE_SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"].rstrip("/")
AZURE_SEARCH_INDEX_NAME = os.environ["AZURE_SEARCH_INDEX_NAME"]
AZURE_SEARCH_API_KEY = os.environ["AZURE_SEARCH_API_KEY"]

AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/") + "/"
AZURE_OPENAI_API_KEY = os.environ["AZURE_OPENAI_API_KEY"]
OPENAI_API_VERSION = os.getenv("OPENAI_API_VERSION", "2024-02-15-preview")
AZURE_OPENAI_CHAT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"]
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]

SEARCH_API_VERSION = os.getenv("AZURE_SEARCH_API_VERSION", "2024-07-01")

search_headers = {
    "Content-Type": "application/json",
    "api-key": AZURE_SEARCH_API_KEY,
}


In [ ]:
def get_search_index_schema() -> dict:
    url = f"{AZURE_SEARCH_ENDPOINT}/indexes/{AZURE_SEARCH_INDEX_NAME}"
    response = requests.get(
        url,
        headers=search_headers,
        params={"api-version": SEARCH_API_VERSION},
        timeout=60,
    )
    response.raise_for_status()
    return response.json()


def pick_content_fields(index_schema: dict) -> list[str]:
    preferred_names = [
        "content",
        "chunk",
        "chunk_text",
        "text",
        "page_content",
        "merged_content",
    ]
    fields = index_schema.get("fields", [])
    searchable_strings = [
        field["name"]
        for field in fields
        if field.get("type") == "Edm.String" and field.get("searchable", False)
    ]

    ordered = [name for name in preferred_names if name in searchable_strings]
    ordered.extend(name for name in searchable_strings if name not in ordered)
    return ordered[:3]


index_schema = get_search_index_schema()
CONTENT_FIELDS = pick_content_fields(index_schema)

if not CONTENT_FIELDS:
    raise ValueError("No searchable string content fields found in the Azure AI Search index.")

print("Using content fields:", CONTENT_FIELDS)


In [ ]:
def retrieve_contexts(question: str, top: int = 4) -> list[str]:
    url = f"{AZURE_SEARCH_ENDPOINT}/indexes/{AZURE_SEARCH_INDEX_NAME}/docs/search"
    payload = {
        "search": question,
        "top": top,
        "queryType": "semantic",
        "select": ",".join(CONTENT_FIELDS),
    }

    response = requests.post(
        url,
        headers=search_headers,
        params={"api-version": SEARCH_API_VERSION},
        json=payload,
        timeout=60,
    )

    if response.status_code == 400:
        payload.pop("queryType", None)
        response = requests.post(
            url,
            headers=search_headers,
            params={"api-version": SEARCH_API_VERSION},
            json=payload,
            timeout=60,
        )

    response.raise_for_status()
    documents = response.json().get("value", [])

    contexts = []
    for doc in documents:
        parts = [str(doc.get(field, "")).strip() for field in CONTENT_FIELDS]
        text = "\n".join(part for part in parts if part)
        if text:
            contexts.append(text)
    return contexts


print(retrieve_contexts("What is self attention?", top=2))


In [ ]:
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

answer_llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=OPENAI_API_VERSION,
    azure_deployment=AZURE_OPENAI_CHAT_DEPLOYMENT,
    temperature=0,
)

judge_llm = answer_llm

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=OPENAI_API_VERSION,
    azure_deployment=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
)


In [ ]:
def answer_with_azure_search(question: str) -> dict:
    contexts = retrieve_contexts(question)
    context_text = "\n\n---\n\n".join(contexts)
    prompt = f"""Answer the question using only the context below.
If the answer is not in the context, say you do not know.

Context:
{context_text}

Question: {question}
"""
    answer = answer_llm.invoke(prompt).content
    return {
        "question": question,
        "answer": answer,
        "contexts": contexts,
    }


sample = answer_with_azure_search("What is self attention?")
print(sample["answer"])
print("Context chunks:", len(sample["contexts"]))


In [ ]:
dataset = [
    {
        "question": "What is self attention?",
        "ground_truth": "Self-attention lets each token in a sequence attend to other tokens in the same sequence to build contextual representations.",
    },
    {
        "question": "Who wrote Attention Is All You Need?",
        "ground_truth": "Attention Is All You Need was written by Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, and Illia Polosukhin.",
    },
]

results = []

for row in dataset:
    result = answer_with_azure_search(row["question"])
    result["ground_truth"] = row["ground_truth"]
    results.append(result)

    print("QUESTION:", result["question"])
    print("ANSWER:", result["answer"][:500], "...")
    print("CONTEXT CHUNKS:", len(result["contexts"]))
    print()


In [ ]:
from datasets import Dataset

eval_dataset = Dataset.from_list(results)
eval_dataset


In [ ]:
from ragas import evaluate
from ragas.metrics import answer_relevancy, context_recall, faithfulness
from ragas.run_config import RunConfig
import asyncio
import threading
import ragas.executor as ragas_executor


def _jupyter_safe_runner_init(
    self,
    jobs,
    desc,
    keep_progress_bar=True,
    raise_exceptions=True,
    run_config=None,
):
    threading.Thread.__init__(self)
    self.jobs = jobs
    self.desc = desc
    self.keep_progress_bar = keep_progress_bar
    self.raise_exceptions = raise_exceptions
    self.run_config = run_config or RunConfig()
    self.loop = None
    self.futures = None
    self.results = None


def _jupyter_safe_runner_run(self):
    self.loop = asyncio.new_event_loop()
    asyncio.set_event_loop(self.loop)
    self.futures = ragas_executor.as_completed(
        loop=self.loop,
        coros=[coro for coro, _ in self.jobs],
        max_workers=self.run_config.max_workers,
    )
    results = []
    try:
        results = self.loop.run_until_complete(self._aresults())
    finally:
        self.results = results
        self.loop.close()


ragas_executor.Runner.__init__ = _jupyter_safe_runner_init
ragas_executor.Runner.run = _jupyter_safe_runner_run

run_config = RunConfig(timeout=180, max_retries=2, max_wait=30, thread_timeout=240)

ragas_result = evaluate(
    eval_dataset,
    metrics=[faithfulness, context_recall, answer_relevancy],
    llm=judge_llm,
    embeddings=embeddings,
    is_async=False,
    run_config=run_config,
)

ragas_result


In [ ]:
ragas_result.to_pandas()
